# Demo 07: Testing neo4j-agent-memory

Este notebook prueba `neo4j-agent-memory` localmente antes de desplegar a AWS.

## Prerequisites

1. **Neo4j running locally:**
   ```bash
   docker run -d \
     --name neo4j-test \
     -p 7474:7474 -p 7687:7687 \
     -e NEO4J_AUTH=neo4j/testpassword \
     neo4j:5.26-community
   ```

2. **Install dependencies:**
   ```bash
   uv pip install -r requirements.txt
   ```

3. **Set environment variables:**
   ```bash
   export NEO4J_URI="bolt://localhost:7687"
   export NEO4J_PASSWORD="testpassword"
   ```

## Setup

In [ ]:
import os
import asyncio
from neo4j_agent_memory import MemoryClient, MemorySettings
from neo4j_agent_memory.config.settings import (
    Neo4jConfig,
    EmbeddingConfig,
    EmbeddingProvider
)

# Configuration
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "testpassword")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

print(f"Connecting to Neo4j at {NEO4J_URI}")

## Test 1: Initialize Memory Client

In [ ]:
settings = MemorySettings(
    neo4j=Neo4jConfig(
        uri=NEO4J_URI,
        user=NEO4J_USER,
        password=NEO4J_PASSWORD,
        database=NEO4J_DATABASE
    ),
    embedding=EmbeddingConfig(
        provider=EmbeddingProvider.SENTENCE_TRANSFORMERS,
        model="all-MiniLM-L6-v2"  # Lightweight model for local testing
    )
)

memory = MemoryClient(settings)

print("✅ Memory client initialized")

## Test 2: Add Conversation Messages (Short-Term Memory)

In [ ]:
async def test_short_term_memory():
    async with memory:
        # Sample user and session IDs (for multi-tenant isolation)
        user_id = "demo-user-001"
        session_id = "demo-session-001"
        
        # Add conversation messages
        await memory.short_term.add_message(
            session_id=session_id,
            role="user",
            content="Hi, I met Sarah Chen from Acme Corp at AWS Summit. She's the VP of Engineering."
        )
        print("✅ Added user message 1")
        
        await memory.short_term.add_message(
            session_id=session_id,
            role="assistant",
            content="Thanks for letting me know! I've saved that Sarah Chen from Acme Corp is someone you met at AWS Summit."
        )
        print("✅ Added assistant message 1")
        
        await memory.short_term.add_message(
            session_id=session_id,
            role="user",
            content="She mentioned they're interested in our enterprise tier for their AI platform project."
        )
        print("✅ Added user message 2")
        
        # Retrieve conversation history
        messages = await memory.short_term.get_messages(session_id=session_id)
        
        print(f"\n📜 Retrieved {len(messages)} messages:")
        for msg in messages:
            print(f"  [{msg.role}] {msg.content[:60]}...")
        
        return session_id

session_id = await test_short_term_memory()

## Test 3: Add Entities (Long-Term Memory with POLE+O)

In [ ]:
async def test_long_term_memory():
    async with memory:
        user_id = "demo-user-001"
        
        # Add Person entity
        sarah = await memory.long_term.add_entity(
            name="Sarah Chen",
            entity_type="Person",
            properties={
                "role": "VP of Engineering",
                "email": "sarah.chen@acmecorp.com",
                "actor_id": user_id  # For multi-tenant isolation
            }
        )
        print(f"✅ Created Person entity: {sarah.name} (ID: {sarah.entity_id})")
        
        # Add Organization entity
        acme = await memory.long_term.add_entity(
            name="Acme Corp",
            entity_type="Organization",
            properties={
                "industry": "Technology",
                "actor_id": user_id
            }
        )
        print(f"✅ Created Organization entity: {acme.name} (ID: {acme.entity_id})")
        
        # Add Event entity
        summit = await memory.long_term.add_entity(
            name="AWS Summit 2026",
            entity_type="Event",
            properties={
                "date": "2026-05-10",
                "location": "San Francisco",
                "actor_id": user_id
            }
        )
        print(f"✅ Created Event entity: {summit.name} (ID: {summit.entity_id})")
        
        return sarah, acme, summit

sarah, acme, summit = await test_long_term_memory()

## Test 4: Add Relationships

In [ ]:
async def test_relationships():
    async with memory:
        # Sarah WORKS_AT Acme
        rel1 = await memory.long_term.add_relationship(
            source_entity_id=sarah.entity_id,
            target_entity_id=acme.entity_id,
            relationship_type="WORKS_AT"
        )
        print(f"✅ Created relationship: Sarah WORKS_AT Acme")
        
        # Sarah ATTENDED Summit
        rel2 = await memory.long_term.add_relationship(
            source_entity_id=sarah.entity_id,
            target_entity_id=summit.entity_id,
            relationship_type="ATTENDED"
        )
        print(f"✅ Created relationship: Sarah ATTENDED AWS Summit")
        
        return rel1, rel2

await test_relationships()

## Test 5: Add User Preferences

In [ ]:
async def test_preferences():
    async with memory:
        # Add preference
        pref = await memory.long_term.add_preference(
            category="product_tier",
            preference="Interested in Enterprise Tier",
            metadata={
                "contact": "Sarah Chen",
                "company": "Acme Corp",
                "project": "AI Platform"
            }
        )
        print(f"✅ Added preference: {pref.preference}")
        
        # Retrieve preferences
        prefs = await memory.long_term.get_preferences(category="product_tier")
        print(f"\n🎯 Retrieved {len(prefs)} preferences:")
        for p in prefs:
            print(f"  - {p.category}: {p.preference}")

await test_preferences()

## Test 6: Add Reasoning Trace (Decision Provenance)

In [ ]:
async def test_reasoning_trace():
    async with memory:
        # Add a decision trace
        trace = await memory.reasoning.add_trace(
            task="Extract entities from conversation",
            steps=[
                {
                    "thought": "User mentioned meeting someone at an event",
                    "action": "extract_entity",
                    "observation": "Found person: Sarah Chen"
                },
                {
                    "thought": "Person has affiliation with organization",
                    "action": "extract_relationship",
                    "observation": "Created WORKS_AT relationship"
                },
                {
                    "thought": "Event reference detected",
                    "action": "extract_entity",
                    "observation": "Found event: AWS Summit 2026"
                }
            ],
            outcome="Successfully extracted 3 entities and 2 relationships from conversation"
        )
        print(f"✅ Added reasoning trace with {len(trace.steps)} steps")
        print(f"   Task: {trace.task}")
        print(f"   Outcome: {trace.outcome}")

await test_reasoning_trace()

## Test 7: Search Context (Semantic Search)

In [ ]:
async def test_search_context():
    async with memory:
        # Search for relevant context
        query = "Who works at Acme Corp?"
        context = await memory.get_context(
            query=query,
            limit=10
        )
        
        print(f"🔍 Query: {query}")
        print(f"\n📊 Context retrieved:")
        print(f"   Messages: {len(context.get('messages', []))}")
        print(f"   Entities: {len(context.get('entities', []))}")
        print(f"   Preferences: {len(context.get('preferences', []))}")
        
        # Display entities found
        if context.get('entities'):
            print(f"\n🏷️  Entities found:")
            for entity in context['entities']:
                print(f"   - {entity.get('name')} ({entity.get('entity_type')})")

await test_search_context()

## Test 8: Query Graph with Cypher

In [ ]:
async def test_cypher_query():
    async with memory:
        # Execute Cypher query
        cypher = """
        MATCH (p:Person)-[r:WORKS_AT]->(o:Organization)
        RETURN p.name AS person, o.name AS organization, r
        """
        
        async with memory.driver.session(database=NEO4J_DATABASE) as session:
            result = await session.run(cypher)
            records = await result.data()
            
            print(f"📊 Cypher Query Results:")
            print(f"   Found {len(records)} relationships\n")
            
            for record in records:
                print(f"   {record['person']} WORKS_AT {record['organization']}")

await test_cypher_query()

## Test 9: Visualize Graph Schema

In [ ]:
async def test_graph_schema():
    async with memory:
        # Get graph schema
        cypher = """
        CALL db.labels() YIELD label
        RETURN collect(label) AS node_labels
        """
        
        async with memory.driver.session(database=NEO4J_DATABASE) as session:
            result = await session.run(cypher)
            record = await result.single()
            labels = record["node_labels"]
            
            print(f"🏷️  Node Labels in Graph:")
            for label in sorted(labels):
                print(f"   - {label}")
        
        # Get relationship types
        cypher = """
        CALL db.relationshipTypes() YIELD relationshipType
        RETURN collect(relationshipType) AS rel_types
        """
        
        async with memory.driver.session(database=NEO4J_DATABASE) as session:
            result = await session.run(cypher)
            record = await result.single()
            rel_types = record["rel_types"]
            
            print(f"\n🔗 Relationship Types:")
            for rel_type in sorted(rel_types):
                print(f"   - {rel_type}")

await test_graph_schema()

## Test 10: Entity Resolution (Deduplication)

In [ ]:
async def test_entity_resolution():
    async with memory:
        user_id = "demo-user-001"
        
        # Try to add Sarah Chen again with slight variation
        sarah_duplicate = await memory.long_term.add_entity(
            name="S. Chen",  # Variation of name
            entity_type="Person",
            properties={
                "role": "VP Engineering",  # Slight variation
                "email": "sarah.chen@acmecorp.com",  # Same email
                "actor_id": user_id
            }
        )
        
        print(f"✅ Added entity (may be deduplicated): {sarah_duplicate.name}")
        print(f"   Entity ID: {sarah_duplicate.entity_id}")
        print(f"   Original Sarah ID: {sarah.entity_id}")
        
        if sarah_duplicate.entity_id == sarah.entity_id:
            print(f"\n✅ Entity resolution worked! Same entity detected.")
        else:
            print(f"\n⚠️  Different entity created (entity resolution may need tuning)")
        
        # Count Person entities
        cypher = "MATCH (p:Person) WHERE p.actor_id = $actor_id RETURN count(p) AS count"
        async with memory.driver.session(database=NEO4J_DATABASE) as session:
            result = await session.run(cypher, actor_id=user_id)
            record = await result.single()
            count = record["count"]
            print(f"\n📊 Total Person entities for user: {count}")

await test_entity_resolution()

## Summary

**✅ All tests completed!**

### What we tested:
1. ✅ Short-term memory (conversation messages)
2. ✅ Long-term memory (POLE+O entities)
3. ✅ Relationships (WORKS_AT, ATTENDED)
4. ✅ User preferences
5. ✅ Reasoning traces (decision provenance)
6. ✅ Semantic search (context retrieval)
7. ✅ Cypher queries
8. ✅ Graph schema inspection
9. ✅ Entity resolution

### Next Steps:

1. **View in Neo4j Browser:**
   - Open http://localhost:7474
   - Login with neo4j/testpassword
   - Run: `MATCH (n)-[r]->(m) RETURN n, r, m LIMIT 50`

2. **Deploy to AWS:**
   ```bash
   cd cdk
   python3 -m venv .venv
   source .venv/bin/activate
   pip install -r requirements.txt
   cd layers && ./build_layer.sh && cd ..
   cdk deploy
   ```

3. **Integrate with Strands:**
   - See `INTEGRATION_PLAN.md` for details
   - Use `context_graph_tools()` from neo4j-agent-memory

## Cleanup

Close memory client connection:

In [ ]:
# Close memory client (if still open)
if memory.driver:
    await memory.driver.close()
    print("✅ Memory client closed")